# 🤖 Mục 4 — Chatbot Giải Thích Risk Drivers
## German Credit Risk Scoring Application

> **Nội dung:** Chatbot tương tác bằng `ipywidgets` giải thích các yếu tố rủi ro, hướng dẫn chọn threshold theo risk appetite, và phân tích từng khách hàng.

---

**Cài đặt thêm nếu cần:**
```
pip install ipywidgets
```

## ⚙️ Bước 1 — Cấu hình & Tìm đường dẫn Data

In [1]:
import os

def find_data_path():
    for base in [os.getcwd(),
                 os.path.join(os.getcwd(), '..'),
                 os.path.join(os.getcwd(), '..', '..'),
                 os.path.join(os.getcwd(), 'Data'),
                 os.path.join(os.getcwd(), '..', 'Data')]:
        for sub in ['Data', '']:
            c = os.path.normpath(os.path.join(base, sub, 'german.data'))
            if os.path.exists(c):
                return c
    return None

DATA_PATH = find_data_path()
if DATA_PATH is None:
    DATA_PATH = r'C:\DAP_PRroject_Final\Data\german.data'
    print(f'[WARNING] Dùng đường dẫn thủ công: {DATA_PATH}')
else:
    print(f'[OK] Data: {DATA_PATH}')
print(f'[OK] Tồn tại: {os.path.exists(DATA_PATH)}')

[OK] Data: c:\DAP_PRroject_Final\Data\german.data
[OK] Tồn tại: True


## 📦 Bước 2 — Import thư viện

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

# Kiểm tra ipywidgets
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_OK = True
    print('✅ ipywidgets: OK — Chatbot UI đầy đủ sẽ được hiển thị')
except ImportError:
    WIDGETS_OK = False
    print('⚠️  ipywidgets chưa cài. Chạy: pip install ipywidgets')
    print('    Sẽ dùng chế độ text đơn giản.')
    from IPython.display import display, HTML

✅ ipywidgets: OK — Chatbot UI đầy đủ sẽ được hiển thị


## 🧠 Bước 3 — Load & Train Model

In [3]:
COLUMNS = [
    'checking', 'duration', 'credit_history', 'purpose', 'amount',
    'savings', 'employment', 'installment_rate', 'personal_status',
    'other_debtors', 'residence_since', 'property', 'age',
    'other_plans', 'housing', 'existing_credits', 'job',
    'dependents', 'telephone', 'foreign_worker', 'label'
]

df_raw  = pd.read_csv(DATA_PATH, sep=' ', header=None, names=COLUMNS)
y_all   = (df_raw['label'] == 2).astype(int)
X_raw   = df_raw.drop('label', axis=1)

cat_cols = [
    'checking', 'credit_history', 'purpose', 'savings', 'employment',
    'personal_status', 'other_debtors', 'property', 'other_plans',
    'housing', 'job', 'telephone', 'foreign_worker'
]
X_enc = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)

num_cols = ['duration', 'amount', 'installment_rate', 'residence_since',
            'age', 'existing_credits', 'dependents']
scaler = StandardScaler()
X_enc[num_cols] = scaler.fit_transform(X_enc[num_cols])

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y_all, test_size=0.2, random_state=42, stratify=y_all
)

model = LogisticRegression(
    C=1.0, class_weight='balanced', max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

FEAT_COLS = X_enc.columns.tolist()
AUC_SCORE = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f'✅ Model đã train xong')
print(f'   ROC-AUC trên test set: {AUC_SCORE:.4f}')
print(f'   Features: {len(FEAT_COLS)}')

✅ Model đã train xong
   ROC-AUC trên test set: 0.8074
   Features: 48


## 📚 Bước 4 — Knowledge Base & Scoring Engine

In [4]:
# ══════════════════════════════════════════════════════
# KNOWLEDGE BASE — Nội dung chatbot trả lời
# ══════════════════════════════════════════════════════

KNOWLEDGE = {
    'threshold': """
<b>⚖️ CHỌN THRESHOLD THEO RISK APPETITE</b><br><br>
<table style='border-collapse:collapse;font-size:13px;width:100%'>
<tr style='background:#2e3350'>
  <th style='padding:6px 10px;text-align:left'>Threshold</th>
  <th style='padding:6px 10px'>Recall Bad</th>
  <th style='padding:6px 10px'>Precision Bad</th>
  <th style='padding:6px 10px'>Profile</th>
</tr>
<tr><td style='padding:5px 10px'>0.25 – 0.30</td><td style='text-align:center'>~88-92%</td><td style='text-align:center'>~38-42%</td><td style='color:#ef4444'>🔴 Max Conservative</td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px'><b>0.35 – 0.40 ⭐</b></td><td style='text-align:center'><b>~74-80%</b></td><td style='text-align:center'><b>~49-54%</b></td><td style='color:#f59e0b'><b>🟡 Optimal Cost-Based</b></td></tr>
<tr><td style='padding:5px 10px'>0.45 – 0.50</td><td style='text-align:center'>~60-65%</td><td style='text-align:center'>~56-60%</td><td style='color:#4f8ef7'>🔵 Balanced (Default)</td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px'>0.55 – 0.65</td><td style='text-align:center'>~42-50%</td><td style='text-align:center'>~64-70%</td><td style='color:#22c55e'>🟢 High Approval Rate</td></tr>
</table><br>
<b>Công thức Expected Cost:</b>  EC = FN × 5 + FP × 1<br>
<b>→ Threshold mặc định 0.50 KHÔNG tối ưu với cost matrix 5:1</b><br>
<b>→ Khuyến nghị: dùng threshold 0.38 để tối thiểu hóa chi phí.</b>
""",

    'drivers': """
<b>🔑 TOP RISK DRIVERS (Feature Importance)</b><br><br>
<table style='border-collapse:collapse;font-size:13px;width:100%'>
<tr style='background:#2e3350'>
  <th style='padding:6px 10px'>#</th>
  <th style='padding:6px 10px;text-align:left'>Feature</th>
  <th style='padding:6px 10px'>Impact</th>
  <th style='padding:6px 10px;text-align:left'>Giải thích</th>
</tr>
<tr><td style='padding:4px 10px;text-align:center'><b>1</b></td><td>Checking Account</td><td style='text-align:center;color:#ef4444'><b>0.92</b></td><td>Số dư âm → tăng rủi ro mạnh nhất</td></tr>
<tr style='background:#1e2235'><td style='padding:4px 10px;text-align:center'><b>2</b></td><td>Duration (tháng)</td><td style='text-align:center;color:#ef4444'><b>0.78</b></td><td>Vay > 36 tháng → default 48% (+3x)</td></tr>
<tr><td style='padding:4px 10px;text-align:center'><b>3</b></td><td>Credit History</td><td style='text-align:center;color:#f59e0b'><b>0.74</b></td><td>Từng trễ hạn → rủi ro tăng cao</td></tr>
<tr style='background:#1e2235'><td style='padding:4px 10px;text-align:center'><b>4</b></td><td>Credit Amount</td><td style='text-align:center;color:#f59e0b'><b>0.63</b></td><td>> 10K DM: default rate 60%</td></tr>
<tr><td style='padding:4px 10px;text-align:center'><b>5</b></td><td>Savings Account</td><td style='text-align:center;color:#4f8ef7'><b>0.61</b></td><td>< 100 DM: 36% vs >1000 DM: 12.5%</td></tr>
<tr style='background:#1e2235'><td style='padding:4px 10px;text-align:center'><b>6</b></td><td>Employment Since</td><td style='text-align:center;color:#4f8ef7'><b>0.45</b></td><td>Thất nghiệp → default 37%</td></tr>
<tr><td style='padding:4px 10px;text-align:center'><b>7</b></td><td>Age</td><td style='text-align:center;color:#22c55e'><b>0.41</b></td><td>19-25 tuổi → default 40.9%</td></tr>
<tr style='background:#1e2235'><td style='padding:4px 10px;text-align:center'><b>8</b></td><td>Loan Purpose</td><td style='text-align:center;color:#22c55e'><b>0.31</b></td><td>Giáo dục 44% vs Đào tạo 11.1%</td></tr>
</table>
""",

    'improve': """
<b>💡 CÁC BƯỚC CẢI THIỆN ĐIỂM TÍN DỤNG</b><br><br>
<div style='line-height:2.0;font-size:13px'>
<span style='color:#ef4444'>⚡ Impact CAO:</span><br>
&nbsp;&nbsp;1. 💳 Nâng số dư tài khoản thanh toán về dương (≥ 0 DM)<br>
&nbsp;&nbsp;2. 💰 Tích lũy tiết kiệm ≥ 500 DM trước khi vay<br><br>
<span style='color:#f59e0b'>🔶 Impact TRUNG:</span><br>
&nbsp;&nbsp;3. 📅 Chọn thời hạn vay ngắn hơn (< 24 tháng nếu có thể)<br>
&nbsp;&nbsp;4. 🏦 Duy trì lịch sử trả đúng hạn liên tục 12+ tháng<br><br>
<span style='color:#22c55e'>✅ Impact NHẸ:</span><br>
&nbsp;&nbsp;5. 💼 Ổn định việc làm ≥ 4 năm liên tục<br>
&nbsp;&nbsp;6. 💵 Giảm số tiền vay — < 3,000 DM: default rate chỉ 25.6%<br>
&nbsp;&nbsp;7. 🎯 Chọn mục đích vay ít rủi ro (Radio/TV 22% vs Giáo dục 44%)
</div>
""",

    'model': """
<b>📊 SO SÁNH MÔ HÌNH</b><br><br>
<table style='border-collapse:collapse;font-size:13px;width:100%'>
<tr style='background:#2e3350'>
  <th style='padding:6px 10px;text-align:left'>Mô hình</th>
  <th style='padding:6px 10px'>Accuracy</th>
  <th style='padding:6px 10px'>ROC-AUC</th>
  <th style='padding:6px 10px'>F1 Bad</th>
  <th style='padding:6px 10px'>Exp. Cost</th>
</tr>
<tr><td style='padding:5px 10px'>Majority Baseline</td><td style='text-align:center'>70.0%</td><td style='text-align:center'>0.500</td><td style='text-align:center;color:#ef4444'>0.000</td><td style='text-align:center;color:#ef4444'>450</td></tr>
<tr style='background:#1e3a5c'><td style='padding:5px 10px'><b>✅ Logistic Reg</b></td><td style='text-align:center'><b>74.5%</b></td><td style='text-align:center;color:#4f8ef7'><b>0.779</b></td><td style='text-align:center;color:#22c55e'><b>0.588</b></td><td style='text-align:center;color:#22c55e'><b>198</b></td></tr>
<tr><td style='padding:5px 10px'>SVM (RBF)</td><td style='text-align:center'>72.0%</td><td style='text-align:center;color:#7c5cbf'>0.761</td><td style='text-align:center;color:#7c5cbf'>0.560</td><td style='text-align:center;color:#7c5cbf'>221</td></tr>
</table><br>
<b>→ Logistic Regression thắng toàn diện:</b><br>
&nbsp;&nbsp;• AUC cao nhất (0.779)<br>
&nbsp;&nbsp;• Expected Cost thấp nhất (198)<br>
&nbsp;&nbsp;• Giải thích được qua coefficients (interpretable)
""",

    'dataset': """
<b>📁 GERMAN CREDIT DATASET</b><br><br>
<div style='font-size:13px;line-height:1.9'>
<b>Nguồn:</b> Prof. Dr. Hans Hofmann, Universität Hamburg<br>
<b>Số mẫu:</b> 1,000 (700 Good + 300 Bad)<br>
<b>Features:</b> 20 (7 numeric + 13 categorical)<br>
<b>Label:</b> 1 = Good Credit | 2 = Bad Credit<br>
<b>Imbalance:</b> 70% Good vs 30% Bad<br><br>
<b>Thống kê chính:</b><br>
• Avg Duration : 20.9 tháng (4–72)<br>
• Avg Amount   : 3,271 DM  (250–18,424)<br>
• Avg Age      : 35.5 tuổi (19–75)<br>
• Default Rate : 30% tổng thể<br><br>
<b>Cost Matrix (5:1):</b><br>
• FN (Bad→Good): cost = 5 (ngân hàng mất tiền)<br>
• FP (Good→Bad): cost = 1 (bỏ lỡ cơ hội)
</div>
""",

    'checking': """
<b>🏦 CHECKING ACCOUNT — Feature Quan Trọng Nhất</b><br><br>
<table style='border-collapse:collapse;font-size:13px;width:100%'>
<tr style='background:#2e3350'><th style='padding:6px 10px'>Mã</th><th style='padding:6px 10px;text-align:left'>Nghĩa</th><th style='padding:6px 10px'>Default Rate</th></tr>
<tr><td style='padding:5px 10px;text-align:center'>A11</td><td>Số dư &lt; 0 DM (Âm)</td><td style='text-align:center;color:#ef4444'><b>~50%+</b></td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px;text-align:center'>A12</td><td>0 – 200 DM</td><td style='text-align:center;color:#f59e0b'><b>~33%</b></td></tr>
<tr><td style='padding:5px 10px;text-align:center'>A13</td><td>≥ 200 DM / lương ≥ 1 năm</td><td style='text-align:center;color:#4f8ef7'><b>~20%</b></td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px;text-align:center'>A14</td><td>Không có tài khoản</td><td style='text-align:center;color:#22c55e'><b>~15%</b></td></tr>
</table><br>
💡 <i>Nghịch lý: Không có tài khoản an toàn hơn tài khoản âm! Lý do: người không có tài khoản thường dùng tiền mặt — ít nợ hơn.</i>
""",

    'duration': """
<b>📅 DURATION — Thời Hạn Vay vs Default Rate</b><br><br>
<table style='border-collapse:collapse;font-size:13px;width:100%'>
<tr style='background:#2e3350'><th style='padding:6px 10px'>Thời hạn</th><th style='padding:6px 10px'>Default Rate</th><th style='padding:6px 10px'>Số mẫu</th><th style='padding:6px 10px'>Rủi ro</th></tr>
<tr><td style='padding:5px 10px'>&lt; 12 tháng</td><td style='text-align:center;color:#22c55e'><b>15.0%</b></td><td style='text-align:center'>180</td><td>🟢 Thấp</td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px'>12 – 24 tháng</td><td style='text-align:center;color:#f59e0b'><b>28.3%</b></td><td style='text-align:center'>406</td><td>🟡 Trung bình</td></tr>
<tr><td style='padding:5px 10px'>24 – 36 tháng</td><td style='text-align:center;color:#f59e0b'><b>31.1%</b></td><td style='text-align:center'>244</td><td>🟡 Trung bình-cao</td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px'>&gt; 36 tháng</td><td style='text-align:center;color:#ef4444'><b>47.9%</b></td><td style='text-align:center'>169</td><td>🔴 Cao (3x rủi ro!)</td></tr>
</table><br>
<b>Lý do:</b> Window thời gian dài → nhiều biến cố tài chính (mất việc, chi phí y tế...) có thể xảy ra. Người vay dài hạn thường có tài chính yếu hơn.
""",

    'savings': """
<b>💰 SAVINGS ACCOUNT — Tiết Kiệm vs Default Rate</b><br><br>
<table style='border-collapse:collapse;font-size:13px;width:100%'>
<tr style='background:#2e3350'><th style='padding:6px 10px'>Mức tiết kiệm</th><th style='padding:6px 10px'>Default Rate</th><th style='padding:6px 10px'>Số người</th></tr>
<tr><td style='padding:5px 10px'>&lt; 100 DM</td><td style='text-align:center;color:#ef4444'><b>36.0%</b></td><td style='text-align:center'>603</td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px'>100 – 500 DM</td><td style='text-align:center;color:#f59e0b'><b>33.0%</b></td><td style='text-align:center'>103</td></tr>
<tr><td style='padding:5px 10px'>Unknown / Không có</td><td style='text-align:center;color:#4f8ef7'><b>17.5%</b></td><td style='text-align:center'>183</td></tr>
<tr style='background:#1e2235'><td style='padding:5px 10px'>500 – 1000 DM</td><td style='text-align:center;color:#4f8ef7'><b>17.5%</b></td><td style='text-align:center'>63</td></tr>
<tr><td style='padding:5px 10px'>&gt; 1000 DM</td><td style='text-align:center;color:#22c55e'><b>12.5%</b></td><td style='text-align:center'>48</td></tr>
</table><br>
✅ <b>Người có tiết kiệm &gt; 1000 DM an toàn hơn 3× so với &lt; 100 DM!</b>
""",
}

# ── Intent Detection ────────────────────────────────────
def detect_intent(msg):
    m = msg.lower()
    intent_map = [
        (['threshold', 'nguong', 'risk appetite', 'conservative', 'chap thuan', 'tu choi', 'nguỡng'], 'threshold'),
        (['driver', 'quan trong', 'feature', 'yeu to', 'importance', 'anh huong', 'ảnh hưởng', 'yếu tố'], 'drivers'),
        (['cai thien', 'cải thiện', 'improve', 'tang diem', 'better', 'tăng điểm', 'tư vấn'], 'improve'),
        (['model', 'mo hinh', 'mô hình', 'logistic', 'svm', 'auc', 'compare', 'so sanh', 'so sánh'], 'model'),
        (['dataset', 'du lieu', 'dữ liệu', 'german', 'bao nhieu', 'bao nhiêu', 'thong tin', 'thông tin'], 'dataset'),
        (['checking', 'tai khoan thanh toan', 'tài khoản', 'account'], 'checking'),
        (['duration', 'thoi han', 'thời hạn', 'thang', 'bao lau', 'bao lâu'], 'duration'),
        (['savings', 'tiet kiem', 'tiết kiệm'], 'savings'),
    ]
    for keywords, intent in intent_map:
        if any(k in m for k in keywords):
            return intent
    return 'general'

def get_response_html(msg):
    intent = detect_intent(msg)
    if intent in KNOWLEDGE:
        return KNOWLEDGE[intent]
    return """
<b>🤖 Tôi có thể trả lời về:</b><br><br>
<div style='line-height:2;font-size:13px'>
• <b>threshold</b>  — Cách chọn ngưỡng phân loại phù hợp risk appetite<br>
• <b>drivers</b>    — Các yếu tố rủi ro quan trọng nhất<br>
• <b>improve</b>    — Cách cải thiện điểm tín dụng<br>
• <b>model</b>      — So sánh Logistic Regression vs SVM<br>
• <b>dataset</b>    — Thông tin về German Credit Dataset<br>
• <b>checking</b>   — Phân tích Checking Account<br>
• <b>duration</b>   — Thời hạn vay vs Rủi ro<br>
• <b>savings</b>    — Tiết kiệm vs Rủi ro<br>
</div>
"""

print('✅ Knowledge base và scoring engine sẵn sàng.')

✅ Knowledge base và scoring engine sẵn sàng.


## 🔢 Bước 5 — Scoring Engine (Dự đoán cho từng khách hàng)

In [5]:
def score_customer(checking, history, purpose, savings, employment,
                   duration, amount, age,
                   installment_rate=2, personal_status='A93',
                   other_debtors='A101', property_='A121',
                   residence=2, existing_credits=1,
                   other_plans='A143', housing='A152',
                   job='A173', dependents=1,
                   telephone='A191', foreign='A201'):
    """Chấm điểm rủi ro tín dụng cho 1 khách hàng."""
    row = {
        'checking': checking, 'duration': duration,
        'credit_history': history, 'purpose': purpose,
        'amount': amount, 'savings': savings, 'employment': employment,
        'installment_rate': installment_rate, 'personal_status': personal_status,
        'other_debtors': other_debtors, 'residence_since': residence,
        'property': property_, 'age': age, 'other_plans': other_plans,
        'housing': housing, 'existing_credits': existing_credits,
        'job': job, 'dependents': dependents,
        'telephone': telephone, 'foreign_worker': foreign,
    }
    df_row = pd.DataFrame([row])
    cat_cols_sc = [
        'checking', 'credit_history', 'purpose', 'savings', 'employment',
        'personal_status', 'other_debtors', 'property', 'other_plans',
        'housing', 'job', 'telephone', 'foreign_worker'
    ]
    df_enc = pd.get_dummies(df_row, columns=cat_cols_sc, drop_first=True)
    for col in FEAT_COLS:
        if col not in df_enc.columns:
            df_enc[col] = 0
    df_enc = df_enc[FEAT_COLS]
    num_cols_sc = ['duration', 'amount', 'installment_rate', 'residence_since',
                   'age', 'existing_credits', 'dependents']
    df_enc[num_cols_sc] = scaler.transform(df_enc[num_cols_sc])
    return float(model.predict_proba(df_enc)[0][1])

# Test nhanh
prob_test = score_customer(
    checking='A11', history='A32', purpose='A43',
    savings='A61', employment='A73', duration=24, amount=3000, age=30
)
print(f'✅ Scoring engine OK. Test probability: {prob_test:.4f}')

✅ Scoring engine OK. Test probability: 0.9455


## 💬 Bước 6 — Chạy Chatbot (ipywidgets UI)

In [ ]:
if WIDGETS_OK:
    chat_history = []

    # ── Style ──────────────────────────────────────────────
    STYLE = """
    <style>
    .chat-container { font-family: 'Segoe UI', sans-serif; }
    .msg-bot  { background:#1e2235; color:#e8eaf6; padding:10px 14px;
                border-radius:4px 12px 12px 12px; margin:6px 0;
                display:inline-block; max-width:85%; font-size:13px;
                border-left:3px solid #4f8ef7; }
    .msg-user { background:#4f8ef7; color:#fff; padding:10px 14px;
                border-radius:12px 4px 12px 12px; margin:6px 0;
                display:inline-block; max-width:75%; font-size:13px; }
    .msg-row-bot  { text-align:left;  margin:4px 0; }
    .msg-row-user { text-align:right; margin:4px 0; }
    .score-bar-wrap { background:#2e3350; border-radius:6px; height:12px; overflow:hidden; margin:6px 0; }
    </style>
    """

    def fmt_msg_html(role, text):
        if role == 'user':
            return f'<div class="msg-row-user"><span class="msg-user">{text}</span></div>'
        else:
            return f'<div class="msg-row-bot"><span class="msg-bot">{text}</span></div>'

    # ── Widgets ────────────────────────────────────────────
    header_html = widgets.HTML(
        f'<div style="background:#1a1d2e;padding:14px 18px;border-radius:10px;margin-bottom:8px">'  
        f'<span style="font-size:18px;font-weight:700;color:#4f8ef7">🤖 Credit Risk Chatbot</span><br>'
        f'<span style="font-size:12px;color:#8892b0">German Credit Model | ROC-AUC: {AUC_SCORE:.4f} | 1,000 mẫu</span>'
        f'</div>'
    )

    chat_out = widgets.Output(
        layout=widgets.Layout(
            border='1px solid #2e3350', height='420px',
            overflow_y='auto', padding='12px',
            background_color='#0f1117', border_radius='8px'
        )
    )

    inp = widgets.Text(
        placeholder='Nhập câu hỏi... (threshold, drivers, improve, model, dataset, checking, duration, savings)',
        layout=widgets.Layout(width='76%', height='36px')
    )
    send_btn  = widgets.Button(description='Gửi ▶', button_style='primary',
                               layout=widgets.Layout(width='11%', height='36px'))
    clear_btn = widgets.Button(description='Xóa 🗑', button_style='warning',
                               layout=widgets.Layout(width='11%', height='36px'))

    quick_buttons = [
        ('Threshold?',     'threshold'),
        ('Risk Drivers?',  'drivers'),
        ('Cải thiện?',     'improve'),
        ('So sánh Model?', 'model'),
        ('Dataset?',       'dataset'),
        ('Checking Acc?',  'checking'),
    ]
    quick_btn_widgets = [
        widgets.Button(description=label, layout=widgets.Layout(width='150px', height='30px'),
                       button_style='info')
        for label, _ in quick_buttons
    ]
    quick_row = widgets.HBox(quick_btn_widgets,
                             layout=widgets.Layout(flex_wrap='wrap', gap='4px'))

    def send_message(msg_text):
        if not msg_text.strip():
            return
        chat_history.append(('user', msg_text))
        response_html = get_response_html(msg_text)
        chat_history.append(('bot', response_html))
        inp.value = ''
        with chat_out:
            clear_output(wait=True)
            all_html = STYLE + '<div class="chat-container">'
            for role, text in chat_history:
                all_html += fmt_msg_html(role, text)
            all_html += '</div>'
            display(HTML(all_html))

    def on_send(b):
        send_message(inp.value)

    def on_clear(b):
        chat_history.clear()
        with chat_out:
            clear_output()

    send_btn.on_click(on_send)
    clear_btn.on_click(on_clear)
    inp.on_submit(on_send)

    for btn, (_, keyword) in zip(quick_btn_widgets, quick_buttons):
        def make_handler(kw):
            def handler(b):
                send_message(kw)
            return handler
        btn.on_click(make_handler(keyword))

    # ── Hiển thị ──────────────────────────────────────────
    display(header_html)
    display(chat_out)
    display(widgets.HBox([inp, send_btn, clear_btn],
                         layout=widgets.Layout(gap='6px', margin='6px 0')))
    display(widgets.HTML('<p style="font-size:11px;color:#8892b0;margin:4px 0">Quick questions:</p>'))
    display(quick_row)

    # Tin nhắn chào
    with chat_out:
        welcome = """
<b>👋 Xin chào! Tôi là Credit Risk Assistant.</b><br><br>
Tôi được train trên <b>German Credit Dataset</b> (1,000 mẫu, 20 features).<br>
Hãy hỏi tôi về:<br>
• Threshold tối ưu theo risk appetite của ngân hàng<br>
• Các yếu tố rủi ro quan trọng nhất (drivers)<br>
• Cách cải thiện điểm tín dụng<br>
• So sánh Logistic Regression vs SVM<br><br>
Hoặc nhấn các nút <b>Quick questions</b> bên dưới! 👇
"""
        display(HTML(STYLE + '<div class="chat-container">' + fmt_msg_html('bot', welcome) + '</div>'))

else:
    # Fallback text mode
    print('\n' + '='*55)
    print('  CHATBOT — TEXT MODE (ipywidgets chưa cài)')
    print('='*55)
    for topic in ['dataset', 'drivers', 'threshold', 'improve', 'model']:
        print(f'\n[TOPIC: {topic.upper()}]')
        # Strip HTML tags
        import re
        text = KNOWLEDGE.get(topic, '')
        text = re.sub(r'<[^>]+>', '', text)
        text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&nbsp;', ' ')
        print(text[:600])
        print('...')

HTML(value='<div style="background:#1a1d2e;padding:14px 18px;border-radius:10px;margin-bottom:8px"><span style…

Output(layout=Layout(border_bottom='1px solid #2e3350', border_left='1px solid #2e3350', border_right='1px sol…

HTML(value='<p style="font-size:11px;color:#8892b0;margin:4px 0">Quick questions:</p>')

## 🎯 Bước 7 — Demo: Chấm Điểm & Phân Tích Khách Hàng Cụ Thể

In [7]:
# ══════════════════════════════════════════════════════════════
# Demo chấm điểm cho 3 khách hàng mẫu
# ══════════════════════════════════════════════════════════════

demo_customers = [
    {
        'name': 'Khách hàng A (Low Risk)',
        'checking': 'A14', 'history': 'A30', 'purpose': 'A41',
        'savings': 'A64', 'employment': 'A75',
        'duration': 12, 'amount': 1500, 'age': 45,
        'description': 'Không có tài khoản âm, tiết kiệm cao, vay xe cũ ngắn hạn, lâu năm'
    },
    {
        'name': 'Khách hàng B (Medium Risk)',
        'checking': 'A12', 'history': 'A32', 'purpose': 'A42',
        'savings': 'A62', 'employment': 'A73',
        'duration': 24, 'amount': 3500, 'age': 32,
        'description': 'Tài khoản 0-200 DM, lịch sử ổn, vay nội thất 24 tháng'
    },
    {
        'name': 'Khách hàng C (High Risk)',
        'checking': 'A11', 'history': 'A33', 'purpose': 'A46',
        'savings': 'A61', 'employment': 'A72',
        'duration': 48, 'amount': 8000, 'age': 22,
        'description': 'Tài khoản âm, từng trễ hạn, vay giáo dục 48 tháng, tiết kiệm thấp'
    },
]

THRESHOLD = 0.38

print('=' * 65)
print('  DEMO: CHẤM ĐIỂM RỦI RO TÍN DỤNG')
print(f'  Threshold: {THRESHOLD} (tối ưu theo cost matrix 5:1)')
print('=' * 65)

for cust in demo_customers:
    prob = score_customer(
        checking=cust['checking'], history=cust['history'],
        purpose=cust['purpose'], savings=cust['savings'],
        employment=cust['employment'], duration=cust['duration'],
        amount=cust['amount'], age=cust['age']
    )
    decision  = '❌ BAD CREDIT — Từ chối' if prob >= THRESHOLD else '✅ GOOD CREDIT — Chấp thuận'
    risk_icon = '🔴' if prob > 0.5 else '🟡' if prob > 0.3 else '🟢'
    bar_len   = int(prob * 30)
    bar_str   = '█' * bar_len + '░' * (30 - bar_len)

    print(f'\n  {cust["name"]}')
    print(f'  {cust["description"]}')
    print(f'  Risk Score : {risk_icon} [{bar_str}] {prob*100:.1f}%')
    print(f'  Threshold  : {THRESHOLD}  →  {decision}')

  DEMO: CHẤM ĐIỂM RỦI RO TÍN DỤNG
  Threshold: 0.38 (tối ưu theo cost matrix 5:1)

  Khách hàng A (Low Risk)
  Không có tài khoản âm, tiết kiệm cao, vay xe cũ ngắn hạn, lâu năm
  Risk Score : 🔴 [███████████████████████████░░░] 90.5%
  Threshold  : 0.38  →  ❌ BAD CREDIT — Từ chối

  Khách hàng B (Medium Risk)
  Tài khoản 0-200 DM, lịch sử ổn, vay nội thất 24 tháng
  Risk Score : 🔴 [████████████████████████████░░] 94.8%
  Threshold  : 0.38  →  ❌ BAD CREDIT — Từ chối

  Khách hàng C (High Risk)
  Tài khoản âm, từng trễ hạn, vay giáo dục 48 tháng, tiết kiệm thấp
  Risk Score : 🔴 [█████████████████████████████░] 98.5%
  Threshold  : 0.38  →  ❌ BAD CREDIT — Từ chối


## 🎛️ Bước 8 — Interactive Scoring Widget

In [ ]:
if WIDGETS_OK:
    # ── Dropdowns & Sliders ───────────────────────────────
    w_checking = widgets.Dropdown(
        options=[
            ('Không có tài khoản (an toàn)', 'A14'),
            ('>= 200 DM / lương >= 1 năm',  'A13'),
            ('0 – 200 DM',                  'A12'),
            ('< 0 DM (Âm — rủi ro cao)',    'A11'),
        ],
        description='Checking:', style={'description_width':'110px'},
        layout=widgets.Layout(width='400px')
    )
    w_history = widgets.Dropdown(
        options=[
            ('Không vay / Đã trả hết',  'A30'),
            ('Đã trả tại ngân hàng',    'A31'),
            ('Đang trả đúng hạn',       'A32'),
            ('Từng trễ hạn',            'A33'),
            ('Tài khoản khó khăn',      'A34'),
        ],
        value='A32',
        description='Lịch sử:', style={'description_width':'110px'},
        layout=widgets.Layout(width='400px')
    )
    w_purpose = widgets.Dropdown(
        options=[
            ('Radio/TV (22.1%)',         'A43'),
            ('Xe cũ (16.5%)',            'A41'),
            ('Đào tạo (11.1%)',          'A48'),
            ('Nội thất (32.0%)',         'A42'),
            ('Kinh doanh (35.1%)',       'A49'),
            ('Xe mới (38.0%)',           'A40'),
            ('Giáo dục (44.0%)',         'A46'),
        ],
        description='Mục đích:', style={'description_width':'110px'},
        layout=widgets.Layout(width='400px')
    )
    w_savings = widgets.Dropdown(
        options=[
            ('> 1000 DM (an toàn)', 'A64'),
            ('500-1000 DM',         'A63'),
            ('Unknown',             'A65'),
            ('100-500 DM',         'A62'),
            ('< 100 DM (rủi ro)',  'A61'),
        ],
        description='Tiết kiệm:', style={'description_width':'110px'},
        layout=widgets.Layout(width='400px')
    )
    w_duration = widgets.IntSlider(
        min=4, max=72, value=24, step=1,
        description='Thời hạn (tháng):', style={'description_width':'130px'},
        layout=widgets.Layout(width='400px')
    )
    w_amount = widgets.IntSlider(
        min=250, max=18000, value=3000, step=250,
        description='Số tiền (DM):', style={'description_width':'130px'},
        layout=widgets.Layout(width='400px')
    )
    w_age = widgets.IntSlider(
        min=19, max=75, value=30, step=1,
        description='Tuổi:', style={'description_width':'130px'},
        layout=widgets.Layout(width='400px')
    )
    w_threshold = widgets.FloatSlider(
        min=0.20, max=0.70, value=0.38, step=0.01,
        description='Threshold:', style={'description_width':'130px'},
        layout=widgets.Layout(width='400px'),
        readout_format='.2f'
    )

    result_out = widgets.Output()

    def update_score(*args):
        prob = score_customer(
            checking=w_checking.value, history=w_history.value,
            purpose=w_purpose.value, savings=w_savings.value,
            employment='A73', duration=w_duration.value,
            amount=w_amount.value, age=w_age.value
        )
        t = w_threshold.value
        decision = '❌ BAD CREDIT — Từ chối' if prob >= t else '✅ GOOD CREDIT — Chấp thuận'
        color = '#ef4444' if prob >= t else '#22c55e'
        bar_pct = prob * 100
        risk_level = 'Cao' if prob > 0.5 else 'Trung bình' if prob > 0.3 else 'Thấp'
        risk_color = '#ef4444' if prob > 0.5 else '#f59e0b' if prob > 0.3 else '#22c55e'

        html_result = f"""
        <div style='background:#1e2235;border:1px solid #2e3350;border-radius:10px;padding:16px;font-family:Segoe UI,sans-serif'>
          <div style='font-size:20px;font-weight:700;color:{color};margin-bottom:8px'>{decision}</div>
          <div style='font-size:13px;color:#8892b0;margin-bottom:12px'>
            Xác suất rủi ro: <b style='color:{risk_color}'>{prob*100:.1f}%</b> &nbsp;|
            Mức rủi ro: <b style='color:{risk_color}'>{risk_level}</b> &nbsp;|
            Threshold: <b style='color:#4f8ef7'>{t:.2f}</b>
          </div>
          <div style='background:#0f1117;border-radius:6px;height:14px;overflow:hidden;margin-bottom:12px'>
            <div style='width:{bar_pct:.1f}%;height:100%;background:{color};border-radius:6px;transition:width 0.3s'></div>
          </div>
          <div style='font-size:12px;color:#8892b0'>⬅ 0% (An toàn) &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 100% (Nguy hiểm) ➡</div>
        </div>
        """
        with result_out:
            clear_output(wait=True)
            display(HTML(html_result))

    for w in [w_checking, w_history, w_purpose, w_savings,
              w_duration, w_amount, w_age, w_threshold]:
        w.observe(update_score, names='value')

    display(widgets.HTML('<h3 style="color:#4f8ef7">🎛️ Interactive Credit Scoring</h3>'))
    display(widgets.HBox([
        widgets.VBox([w_checking, w_history, w_purpose, w_savings]),
        widgets.VBox([w_duration, w_amount, w_age, w_threshold])
    ]))
    display(result_out)
    update_score()  # Initial calculation

else:
    print('\n[INFO] ipywidgets chưa cài. Chạy: pip install ipywidgets')
    print('Chạy demo tĩnh bên trên (Bước 7) để xem kết quả.')

HTML(value='<h3 style="color:#4f8ef7">🎛️ Interactive Credit Scoring</h3>')

Output()